# M22 Discovery Entrypoint — Telco Dataset Authoring Notebook

Telco Customer Churn authoring entrypoint for Atlas DataFlow (Project Spec S0011/S0012/S0013/S0014/S0015/S0016).

This notebook is the Telco-specific authoring surface for
`data/raw/telco-customer-churn.csv`. It loads the raw CSV, verifies known
structural facts, records authoring observations, and assembles a narrow
`dataset_modeling_intent.v1` authoring contract (Project Spec S0013) from
those observations, then projects that modeling intent into an
`execution_contract_draft.v1` candidate (Project Spec S0014), reports
governed training invocation readiness (Project Spec S0015), and reports
release-candidate data handoff readiness (Project Spec S0016). Common
loading, inspection, modeling-intent-building, and draft-projection logic
is implemented once as reusable, tested helper functions in
`pipeline/discovery_evidence.py` (Project Specs S0012/S0013),
`pipeline/contract_derivation.py` (Project Spec S0014), `pipeline/training.py`
(Project Spec S0015), and `pipeline/assemble_candidate.py` (Project Spec
S0016); this notebook calls those helpers and keeps only Telco-specific
decisions (expected structural facts, target column, identifier columns,
feature review notes) visible inline.

## Boundaries

- This notebook is an **authoring notebook only**.
- Notebook output is **not** the final operational source of truth.
- This notebook does not train models, select model families, create release
  candidates, validate publisher candidates, promote releases, mutate
  registry state, or change API/UI behavior. A release-candidate data
  handoff readiness check is not a release-candidate assembly, publisher
  validation, publisher promotion, registry activation, API availability,
  or UI data-fill.
- Any local output produced by this notebook is a **non-promoted authoring
  artifact**. It must not be treated as an official contract, release
  candidate, publisher run, registry file, model binary, or UI data fixture.
- Downstream contract derivation, model training, and publication are
  separate, later stages driven by separate, explicitly authorized specs.

## Usage

Run locally with the default repository-relative parameters, or override
`repo_root` explicitly (for example under papermill) if the notebook is
executed from outside the repository working directory:

```
papermill notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb output.ipynb \
    -p repo_root /path/to/atlas-dataflow
```

Do not rely on implicit notebook state or hidden local paths as inputs to
later pipeline stages.

In [23]:
# Side-effect boundary — these operations are explicitly forbidden in this
# authoring entrypoint. This is checked before any dataset loading occurs.
FORBIDDEN_SIDE_EFFECTS = {
    "model_training": False,
    "model_family_selection": False,
    "release_candidate_creation": False,
    "publisher_run_creation": False,
    "release_promotion": False,
    "registry_state_mutation": False,
    "api_behavior_change": False,
    "ui_behavior_change": False,
    "reusable_helper_module_creation": False,
    "notebook_output_committed_as_official_artifact": False,
    "modeling_intent_treated_as_execution_contract": False,
    "execution_contract_draft_promoted_to_official_contract": False,
    "release_candidate_handoff_readiness_treated_as_assembled_candidate": False,
    "release_candidate_handoff_derived_from_notebook_state": False,
    "prepared_data_metadata_treated_as_final_training_input": False,
    "execution_contract_blank_value_policy_silently_approved": False,
    "execution_contract_derived_from_hidden_notebook_state": False,
    "training_run_materialized_from_raw_dataset_or_notebook_state": False,
    "model_card_conversion_mixed_with_public_release_or_profile_publication": False,
}
assert all(not v for v in FORBIDDEN_SIDE_EFFECTS.values()), (
    "Side-effect boundary violated; this entrypoint must not perform any "
    "forbidden operation."
)
print("Side-effect boundaries confirmed:", FORBIDDEN_SIDE_EFFECTS)

Side-effect boundaries confirmed: {'model_training': False, 'model_family_selection': False, 'release_candidate_creation': False, 'publisher_run_creation': False, 'release_promotion': False, 'registry_state_mutation': False, 'api_behavior_change': False, 'ui_behavior_change': False, 'reusable_helper_module_creation': False, 'notebook_output_committed_as_official_artifact': False, 'modeling_intent_treated_as_execution_contract': False, 'execution_contract_draft_promoted_to_official_contract': False, 'release_candidate_handoff_readiness_treated_as_assembled_candidate': False, 'release_candidate_handoff_derived_from_notebook_state': False, 'prepared_data_metadata_treated_as_final_training_input': False, 'execution_contract_blank_value_policy_silently_approved': False, 'execution_contract_derived_from_hidden_notebook_state': False, 'training_run_materialized_from_raw_dataset_or_notebook_state': False, 'model_card_conversion_mixed_with_public_release_or_profile_publication': False}


## Parameters and repository-relative dataset path

`dataset_relative_path` is repository-relative and explicit — no implicit or
hidden absolute paths. `repo_root` defaults to the current working directory
(the expected convention when running this notebook from the repository
root) and may be overridden explicitly, for example by papermill, when the
notebook is executed from elsewhere.

In [24]:
# Parameters — supply explicit overrides here or via papermill; the defaults
# below are the Telco authoring defaults for this notebook.
dataset_slug = "telco-customer-churn"          # Fixed authoring identity for this notebook.
dataset_relative_path = "data/raw/telco-customer-churn.csv"  # Repository-relative, explicit.
repo_root = None                                # Optional override (str); None = current working directory.
target_column = "Churn"                         # Observed target column for this dataset.

## Raw CSV loading

Resolve the repository-relative dataset path against a deterministic Atlas
repository root. When `repo_root` is not supplied, the notebook walks upward
from the current working directory until stable repository markers are found,
so opening the notebook from its nested dataset directory still resolves
repository-relative paths correctly. The raw CSV is loaded with the standard
library `csv` module only. This notebook intentionally
avoids new repository dependencies (for example `pandas`) — none are
authorized by this spec.

In [25]:
import sys
import json
from pathlib import Path

NOTEBOOK_RELATIVE_PATH = Path("notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb")
ATLAS_REPO_ROOT_MARKERS = (
    Path("pipeline/assemble_candidate.py"),
    Path("pipeline/training.py"),
    Path("contracts/telco-customer-churn/dataset-context.json"),
    NOTEBOOK_RELATIVE_PATH,
)


def discover_atlas_repo_root(explicit_repo_root=None):
    if explicit_repo_root:
        candidate = Path(explicit_repo_root).expanduser().resolve()
        if all((candidate / marker).exists() for marker in ATLAS_REPO_ROOT_MARKERS):
            return candidate
        raise FileNotFoundError(
            f"repo_root '{candidate}' does not contain the expected Atlas root markers: "
            f"{[str(marker) for marker in ATLAS_REPO_ROOT_MARKERS]}"
        )

    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in ATLAS_REPO_ROOT_MARKERS):
            return candidate

    raise FileNotFoundError(
        f"Could not discover the Atlas repository root while walking upward from '{start}'. "
        "Supply repo_root explicitly when running outside the atlas-dataflow repository."
    )


_repo_root_path = discover_atlas_repo_root(repo_root)
if str(_repo_root_path) not in sys.path:
    sys.path.insert(0, str(_repo_root_path))

from pipeline.discovery_evidence import (
    materialize_discovery_evidence,
    resolve_repository_path,
    load_dataset_csv,
    summarize_structure,
    observe_authoring_fields,
    summarize_target_column,
    summarize_identifier_columns,
    derive_feature_candidates,
    authoring_helper_evidence_policy,
    build_dataset_modeling_intent,
    materialize_dataset_modeling_intent,
)
from pipeline.prepare_candidate import (
    materialize_verified_conditional_fill_prepared_dataset,
)

dataset_path = resolve_repository_path(dataset_relative_path, repo_root=_repo_root_path)

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {dataset_path}. "
        f"Expected the official raw CSV at repository-relative path "
        f"'{dataset_relative_path}' under repo_root '{_repo_root_path}'. "
        "Supply an explicit repo_root override if running outside the "
        "atlas-dataflow repository."
    )

raw_rows = load_dataset_csv(dataset_path)
structure = summarize_structure(raw_rows)
raw_columns = structure["ordered_columns"]

print(f"dataset_slug   : {dataset_slug}")
print(f"dataset_path   : {dataset_path}")
print(f"row_count      : {structure['row_count']}")
print(f"column_count   : {structure['column_count']}")

dataset_slug   : telco-customer-churn
dataset_path   : /home/fabyuu/Projetos/N8N/atlas-dataflow/data/raw/telco-customer-churn.csv
row_count      : 7043
column_count   : 21


## Structural verification

Verify the row count, column count, ordered column list, and per-field
authoring observations (inferred type, blank string count, null-like count,
cardinality, reduced sample bounds) via the reusable
`observe_authoring_fields` helper from `pipeline/discovery_evidence.py`.
These are structural facts recorded by this Project Spec (S0011/S0012)
against the committed source file. A mismatch means the source CSV changed
since this notebook's observations were authored and must be reviewed
explicitly before trusting the rest of this notebook's recorded
observations.

In [26]:
EXPECTED_ROW_COUNT = 7043
EXPECTED_COLUMN_COUNT = 21
EXPECTED_COLUMNS = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents",
    "tenure", "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling",
    "PaymentMethod", "MonthlyCharges", "TotalCharges", "Churn",
]

assert structure["row_count"] == EXPECTED_ROW_COUNT, (
    f"Row count {structure['row_count']} does not match the recorded authoring "
    f"observation ({EXPECTED_ROW_COUNT}). The source CSV may have changed; "
    "revisit this notebook's recorded structural observations before "
    "trusting downstream sections."
)
assert structure["column_count"] == EXPECTED_COLUMN_COUNT, (
    f"Column count {structure['column_count']} does not match the recorded "
    f"authoring observation ({EXPECTED_COLUMN_COUNT})."
)
assert raw_columns == EXPECTED_COLUMNS, (
    "Ordered column list does not match the recorded authoring observation. "
    f"Observed: {raw_columns}"
)

field_observations = observe_authoring_fields(raw_rows, raw_columns)

print("Structural verification passed: row_count, column_count, ordered columns match.")
print("Field observations (inferred_type, blank_string_count, null_like_count, cardinality):")
print(json.dumps(
    [
        {k: v for k, v in obs.items() if k != "reduced_sample_values"}
        for obs in field_observations
    ],
    indent=2,
))

Structural verification passed: row_count, column_count, ordered columns match.
Field observations (inferred_type, blank_string_count, null_like_count, cardinality):
[
  {
    "name": "customerID",
    "inferred_type": "string",
    "blank_string_count": 0,
    "null_like_count": 0,
    "cardinality": 7043
  },
  {
    "name": "gender",
    "inferred_type": "string",
    "blank_string_count": 0,
    "null_like_count": 0,
    "cardinality": 2
  },
  {
    "name": "SeniorCitizen",
    "inferred_type": "boolean",
    "blank_string_count": 0,
    "null_like_count": 0,
    "cardinality": 2
  },
  {
    "name": "Partner",
    "inferred_type": "boolean",
    "blank_string_count": 0,
    "null_like_count": 0,
    "cardinality": 2
  },
  {
    "name": "Dependents",
    "inferred_type": "boolean",
    "blank_string_count": 0,
    "null_like_count": 0,
    "cardinality": 2
  },
  {
    "name": "tenure",
    "inferred_type": "integer",
    "blank_string_count": 0,
    "null_like_count": 0,
    "ca

## Target-column inspection — `Churn`

Inspect the observed target column labels and distribution via the reusable
`summarize_target_column` helper. `Churn` is the binary target column for
this dataset. The likely positive-class candidate for later modeling review
is `Yes`, but the final positive-label decision must be externalized by a
later modeling-intent contract spec — this notebook only records the
observation, it does not decide it.

In [27]:
EXPECTED_TARGET_LABELS = {"No", "Yes"}
EXPECTED_TARGET_DISTRIBUTION = {"No": 5174, "Yes": 1869}
LIKELY_POSITIVE_CLASS_CANDIDATE = "Yes"  # Observation only — not a final decision.

target_summary = summarize_target_column(raw_rows, target_column)
target_label_set = set(target_summary["observed_labels"])
target_distribution = target_summary["observed_distribution"]

assert target_label_set == EXPECTED_TARGET_LABELS, (
    f"Observed target labels {sorted(target_label_set)} do not match the "
    f"recorded authoring observation {sorted(EXPECTED_TARGET_LABELS)}."
)
assert target_distribution == EXPECTED_TARGET_DISTRIBUTION, (
    f"Observed target distribution {target_distribution} does not match the "
    f"recorded authoring observation {EXPECTED_TARGET_DISTRIBUTION}. The "
    "source CSV may have changed; revisit this notebook's recorded "
    "observations before trusting downstream sections."
)
assert target_summary["is_authoritative"] is False, (
    "summarize_target_column must always report is_authoritative: False."
)

print(f"target_column                  : {target_column}")
print(f"observed_target_labels         : {sorted(target_label_set)}")
print(f"observed_target_distribution   : {target_distribution}")
print(f"likely_positive_class_candidate: {LIKELY_POSITIVE_CLASS_CANDIDATE} (observation only, not a final decision)")

target_column                  : Churn
observed_target_labels         : ['No', 'Yes']
observed_target_distribution   : {'No': 5174, 'Yes': 1869}
likely_positive_class_candidate: Yes (observation only, not a final decision)


## Identifier-column inspection — `customerID`

`customerID` is a per-row identifier candidate, not a modeling feature
candidate, inspected via the reusable `summarize_identifier_columns` helper.
It must be excluded from initial feature candidates without an explicit
override recorded elsewhere.

In [28]:
IDENTIFIER_COLUMNS = ["customerID"]

identifier_summaries = summarize_identifier_columns(raw_rows, IDENTIFIER_COLUMNS)
customer_id_summary = identifier_summaries[0]

assert customer_id_summary["is_unique_per_row"], (
    f"Expected 'customerID' to be unique per row ({customer_id_summary['row_count']} rows), "
    f"observed {customer_id_summary['unique_count']} unique values. An identifier "
    "candidate that is not unique per row must be reviewed explicitly "
    "before being treated as an identifier."
)

print(f"identifier_columns       : {IDENTIFIER_COLUMNS}")
print(f"customerID_unique_count  : {customer_id_summary['unique_count']} (of {customer_id_summary['row_count']} rows)")
print("customerID is recorded as an identifier candidate and is excluded from "
      "initial feature candidates without explicit override.")

identifier_columns       : ['customerID']
customerID_unique_count  : 7043 (of 7043 rows)
customerID is recorded as an identifier candidate and is excluded from initial feature candidates without explicit override.


## Missing and blank-value inspection — `TotalCharges`

Surface the `TotalCharges` blank-string condition explicitly rather than
silently coercing it to numeric. Blank-value handling for `TotalCharges`
must be decided explicitly by a later modeling-intent contract spec before
final modeling intent — this notebook only records the observation.

In [29]:
total_charges_obs = next(o for o in field_observations if o["name"] == "TotalCharges")
total_charges_blank_count = total_charges_obs["blank_string_count"]
total_charges_blank_tenure_values = sorted(
    {row["tenure"] for row in raw_rows if row["TotalCharges"].strip() == ""}
)

if total_charges_blank_count == 0:
    print(
        "No blank 'TotalCharges' values observed in this run of the CSV. "
        "This differs from this notebook's recorded authoring observation "
        "(11 blank values, all at tenure == '0') — revisit before trusting "
        "downstream authoring notes."
    )
else:
    print(f"total_charges_blank_count           : {total_charges_blank_count}")
    print(f"total_charges_blank_tenure_values   : {total_charges_blank_tenure_values}")

print(
    "'TotalCharges' must not be silently treated as fully numeric. Blank-value "
    "handling (for example impute-as-zero, drop, or a distinct missing-value "
    "indicator) must be decided explicitly by a later modeling-intent "
    "contract spec, not implicitly by this authoring notebook or by "
    "downstream training code."
)

total_charges_blank_count           : 11
total_charges_blank_tenure_values   : ['0']
'TotalCharges' must not be silently treated as fully numeric. Blank-value handling (for example impute-as-zero, drop, or a distinct missing-value indicator) must be decided explicitly by a later modeling-intent contract spec, not implicitly by this authoring notebook or by downstream training code.


## Feature-candidate overview

List non-target, non-identifier columns as initial feature candidates only,
via the reusable `derive_feature_candidates` helper. `SeniorCitizen` is
recorded separately as a binary numeric indicator, since it is already
represented as `0`/`1` in the raw CSV rather than as a raw categorical label.

In [30]:
feature_candidate_columns = derive_feature_candidates(
    raw_columns, target_column=target_column, identifier_columns=IDENTIFIER_COLUMNS
)

senior_citizen_values = sorted({row["SeniorCitizen"] for row in raw_rows})
assert set(senior_citizen_values) == {"0", "1"}, (
    f"Expected 'SeniorCitizen' to be a binary numeric indicator with "
    f"observed values {{'0', '1'}}, observed {senior_citizen_values}."
)

excluded_from_feature_candidates = sorted(set(IDENTIFIER_COLUMNS) | {target_column})
print(f"excluded_from_feature_candidates : {excluded_from_feature_candidates}")
print(f"feature_candidate_columns        : {feature_candidate_columns}")
print(f"SeniorCitizen_observed_values    : {senior_citizen_values} (binary numeric indicator)")
print(
    "These are initial feature candidates only — final feature selection, "
    "encoding, and missing-value policy are decided by a later "
    "modeling-intent contract spec, not by this authoring notebook."
)

excluded_from_feature_candidates : ['Churn', 'customerID']
feature_candidate_columns        : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
SeniorCitizen_observed_values    : ['0', '1'] (binary numeric indicator)
These are initial feature candidates only — final feature selection, encoding, and missing-value policy are decided by a later modeling-intent contract spec, not by this authoring notebook.


## Preliminary authoring observations

Assemble the authoring observations recorded by this notebook, plus the
reusable helper's reduced evidence policy confirmation
(`authoring_helper_evidence_policy`), into a single structure. This is an
authoring-time observation record intended to feed a later, separately
authorized modeling-intent contract spec — it is not itself an execution
contract, release candidate, or any other governed pipeline artifact.

In [31]:
authoring_observations = {
    "dataset_slug": dataset_slug,
    "dataset_relative_path": dataset_relative_path,
    "row_count": structure["row_count"],
    "column_count": structure["column_count"],
    "ordered_columns": structure["ordered_columns"],
    "field_observations": field_observations,
    "target_column": target_column,
    "observed_target_labels": sorted(target_label_set),
    "observed_target_distribution": target_distribution,
    "likely_positive_class_candidate": LIKELY_POSITIVE_CLASS_CANDIDATE,
    "positive_class_decision_finalized": False,
    "identifier_columns": IDENTIFIER_COLUMNS,
    "total_charges_blank_count": total_charges_blank_count,
    "total_charges_blank_value_policy_decided": True,  # Resolved by Project Spec S0028.
    "feature_candidate_columns": feature_candidate_columns,
    "senior_citizen_observed_values": senior_citizen_values,
    "reduced_evidence_policy": authoring_helper_evidence_policy(),
    "notebook_boundary": "authoring_notebook_only",
    "notebook_output_is_not_final_operational_truth": True,
    "requires_later_modeling_intent_contract_spec": True,
}

print("Authoring observations:")
print(json.dumps(authoring_observations, indent=2))


Authoring observations:
{
  "dataset_slug": "telco-customer-churn",
  "dataset_relative_path": "data/raw/telco-customer-churn.csv",
  "row_count": 7043,
  "column_count": 21,
  "ordered_columns": [
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn"
  ],
  "field_observations": [
    {
      "name": "customerID",
      "inferred_type": "string",
      "blank_string_count": 0,
      "null_like_count": 0,
      "cardinality": 7043,
      "reduced_sample_values": [
        "0002-ORFBO",
        "0003-MKNFE",
        "0004-TLHLJ",
        "0011-IGKFF",
        "0013-EXCHZ"
      ]
    },
    {
      "name": "gender",
      "inferred_type": "s

## Dataset modeling intent — `dataset_modeling_intent.v1` (Project Spec S0013)

Assemble the reviewed authoring decisions above into a narrow
`dataset_modeling_intent.v1` object via the reusable
`build_dataset_modeling_intent` helper from `pipeline/discovery_evidence.py`.
This is an authoring-intent contract only — it is not an execution contract,
runtime contract, public contract, release candidate input, publisher input,
registry artifact, API fixture, or UI fixture, and it is kept separate from
all of those. `TotalCharges` is recorded as a preparation review item pending
explicit blank-value handling, and `SeniorCitizen` is recorded as requiring
explicit semantic type intent, since its raw representation is numeric but
its semantic domain is binary. No model training, release assembly,
publication, registry mutation, API change, UI change, or notebook execution
is authorized by this section.

In [32]:
MODELING_INTENT_FEATURE_REVIEW_NOTES = {
    "TotalCharges": (
        "Blank-value policy resolved (Project Spec S0028): null/empty/whitespace-only "
        "values verified against 'tenure' == 0 for all 11 observed blank rows, then "
        "filled with 0.0 and normalized to numeric. Downstream execution-contract "
        "regeneration to move TotalCharges from ignored_columns into feature_columns is "
        "out of scope for this policy resolution and requires a separate, explicitly "
        "authorized implementation request."
    ),
    "SeniorCitizen": (
        "Raw representation is numeric (0/1) but the semantic domain is "
        "binary; requires explicit type intent before encoding."
    ),
}
MODELING_INTENT_FEATURE_TYPE_OVERRIDES = {
    "SeniorCitizen": "requires_review",
}
MODELING_INTENT_BLANK_VALUE_POLICY_CANDIDATES = {
    "TotalCharges": "unresolved_pending_review",  # Re-derived from the preparation
    # recipe's own review_status by materialize_dataset_modeling_intent below; this
    # input value is only the fallback used if no preparation recipe is present yet.
}
MODELING_INTENT_OPEN_QUESTIONS = [
    "Final 'SeniorCitizen' semantic type/encoding treatment is not yet "
    "decided.",
    "Final positive-label decision for 'Churn' requires explicit "
    "confirmation despite the observed candidate recorded here.",
]

dataset_modeling_intent = build_dataset_modeling_intent(
    dataset_slug=dataset_slug,
    dataset_source_ref=dataset_relative_path,
    authoring_notebook_ref="notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb",
    columns=raw_columns,
    target_column=target_column,
    task_type="binary_classification",
    observed_labels=sorted(target_label_set),
    positive_label_candidate=LIKELY_POSITIVE_CLASS_CANDIDATE,
    observed_target_distribution=target_distribution,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_review_notes=MODELING_INTENT_FEATURE_REVIEW_NOTES,
    feature_type_intent_overrides=MODELING_INTENT_FEATURE_TYPE_OVERRIDES,
    blank_value_policy_candidates=MODELING_INTENT_BLANK_VALUE_POLICY_CANDIDATES,
    open_questions=MODELING_INTENT_OPEN_QUESTIONS,
)

assert dataset_modeling_intent["contract_version"] == "dataset_modeling_intent.v1"
assert dataset_modeling_intent["target_intent"]["is_final_training_configuration"] is False
assert not any(dataset_modeling_intent["modeling_intent_boundary_confirmations"].values()), (
    "dataset_modeling_intent must not be treated as an execution/runtime/public "
    "contract, release candidate input, publisher input, registry artifact, "
    "API fixture, or UI fixture."
)

print("Dataset modeling intent (draft authoring evidence — not an execution contract):")
print(json.dumps(dataset_modeling_intent, indent=2))


Dataset modeling intent (draft authoring evidence — not an execution contract):
{
  "artifact_type": "dataset_modeling_intent",
  "contract_version": "dataset_modeling_intent.v1",
  "dataset_identity": {
    "dataset_slug": "telco-customer-churn",
    "dataset_source_ref": "data/raw/telco-customer-churn.csv"
  },
  "authoring_source": {
    "authoring_notebook_ref": "notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb",
    "reduced_discovery_evidence_ref": null
  },
  "target_intent": {
    "target_column": "Churn",
    "task_type": "binary_classification",
    "observed_labels": [
      "No",
      "Yes"
    ],
    "positive_label_candidate": "Yes",
    "observed_target_distribution": {
      "No": 5174,
      "Yes": 1869
    },
    "is_final_training_configuration": false
  },
  "identifier_and_ignored_columns": [
    {
      "name": "customerID",
      "reason": "identifier_candidate_excluded_from_features"
    }
  ],
  "initial_feature_candidates": [
    "gender",
 

## Execution contract draft projection — `execution_contract_draft.v1` (Project Spec S0014)

Project the `dataset_modeling_intent.v1` object assembled above into a narrow
`execution_contract_draft.v1` candidate via the reusable
`project_execution_contract_draft` helper from
`pipeline/contract_derivation.py`. This draft reuses `execution_contract.v1`
vocabulary (`target_column`, `feature_columns`, `ignored_columns`) where the
modeling intent already supports it, but it is not an execution contract,
runtime contract, public contract, release candidate input, publisher
input, registry artifact, API fixture, or UI fixture — `TotalCharges`
blank-value handling and `SeniorCitizen` semantic type intent remain
explicit, unresolved review items rather than accepted execution policy,
and `execution_readiness.is_execution_ready` stays `False` since this
projection never supplies final preprocessing, split, metric, or
model-family policy. No model training, release assembly, publication,
registry mutation, API change, UI change, or notebook execution is
authorized by this section.

In [33]:
from pipeline.contract_derivation import project_execution_contract_draft

execution_contract_draft = project_execution_contract_draft(dataset_modeling_intent)

assert execution_contract_draft["artifact_type"] == "execution_contract_draft"
assert execution_contract_draft["contract_version"] != "execution_contract.v1"
assert execution_contract_draft["draft_status"] == "not_execution_ready"
assert execution_contract_draft["execution_readiness"]["is_execution_ready"] is False
assert "customerID" not in execution_contract_draft["feature_columns"]
assert not any(
    execution_contract_draft["execution_contract_draft_boundary_confirmations"].values()
), (
    "execution_contract_draft must not be treated as an execution/runtime/public "
    "contract, release candidate input, publisher input, registry artifact, "
    "API fixture, or UI fixture."
)

print("Execution contract draft candidate (not execution-ready — review required):")
print(json.dumps(execution_contract_draft, indent=2))
print()
print("Blocking reasons before this draft could be promoted to an official "
      "execution contract:")
for reason in execution_contract_draft["execution_readiness"]["blocking_reasons"]:
    print(f"  - {reason}")

Execution contract draft candidate (not execution-ready — review required):
{
  "artifact_type": "execution_contract_draft",
  "contract_version": "execution_contract_draft.v1",
  "draft_status": "not_execution_ready",
  "dataset_identity": {
    "dataset_slug": "telco-customer-churn",
    "dataset_source_ref": "data/raw/telco-customer-churn.csv"
  },
  "authoring_traceability": {
    "authoring_notebook_ref": "notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb",
    "reduced_discovery_evidence_ref": null,
    "source_modeling_intent_contract_version": "dataset_modeling_intent.v1",
    "source_modeling_intent_generated_at": "2026-07-10T19:17:32+00:00"
  },
  "task_intent": {
    "task_type": "binary_classification"
  },
  "target_column": "Churn",
  "target_intent": {
    "target_column": "Churn",
    "observed_labels": [
      "No",
      "Yes"
    ],
    "observed_target_distribution": {
      "No": 5174,
      "Yes": 1869
    },
    "positive_label_candidate": "Yes"


## Non-promoted local output summary

This notebook may optionally write `authoring_observations`,
`dataset_modeling_intent`, and `execution_contract_draft` to local files for
the operator's own convenience during authoring. Any such file is a **non-promoted, draft authoring
artifact** — it lives under `data/`, which is excluded from version control
by this repository's `.gitignore`, and it must not be treated as an official
contract, execution contract, release candidate, publisher run, registry
file, or UI data fixture. Promoting any of these observations into an
official artifact requires a separate, explicitly authorized implementation
request.

In [34]:
write_local_output = False  # Set True locally to write the non-promoted summary files below.

local_output_dir = _repo_root_path / "data" / "raw" / "telco-customer-churn-authoring-output"
local_output_path = local_output_dir / f"{dataset_slug}-authoring-observations.json"
local_modeling_intent_output_path = local_output_dir / f"{dataset_slug}-modeling-intent.draft.json"
local_execution_contract_draft_output_path = local_output_dir / f"{dataset_slug}-execution-contract.draft.json"

if write_local_output:
    local_output_dir.mkdir(parents=True, exist_ok=True)
    local_output_path.write_text(
        json.dumps(authoring_observations, indent=2), encoding="utf-8"
    )
    local_modeling_intent_output_path.write_text(
        json.dumps(dataset_modeling_intent, indent=2), encoding="utf-8"
    )
    local_execution_contract_draft_output_path.write_text(
        json.dumps(execution_contract_draft, indent=2), encoding="utf-8"
    )
    print(f"Non-promoted authoring output written to: {local_output_path}")
    print(
        "Non-promoted, draft dataset_modeling_intent output written to: "
        f"{local_modeling_intent_output_path}"
    )
    print(
        "Non-promoted, draft execution_contract_draft output written to: "
        f"{local_execution_contract_draft_output_path}"
    )
else:
    print(
        "write_local_output is False; no local output files were written. "
        f"If enabled, the non-promoted authoring output would be written to: "
        f"{local_output_path}\n"
        "and the non-promoted, draft modeling-intent output would be written to: "
        f"{local_modeling_intent_output_path}"
        "\nand the non-promoted, draft execution-contract-draft output would "
        f"be written to: {local_execution_contract_draft_output_path}"
    )

print(
    "These output paths are local and non-promoted. Neither is an official "
    "contract, execution contract, release candidate, publisher run, "
    "registry file, or UI data fixture. Promotion requires a separate, "
    "explicitly authorized implementation request."
)

write_local_output is False; no local output files were written. If enabled, the non-promoted authoring output would be written to: /home/fabyuu/Projetos/N8N/atlas-dataflow/data/raw/telco-customer-churn-authoring-output/telco-customer-churn-authoring-observations.json
and the non-promoted, draft modeling-intent output would be written to: /home/fabyuu/Projetos/N8N/atlas-dataflow/data/raw/telco-customer-churn-authoring-output/telco-customer-churn-modeling-intent.draft.json
and the non-promoted, draft execution-contract-draft output would be written to: /home/fabyuu/Projetos/N8N/atlas-dataflow/data/raw/telco-customer-churn-authoring-output/telco-customer-churn-execution-contract.draft.json
These output paths are local and non-promoted. Neither is an official contract, execution contract, release candidate, publisher run, registry file, or UI data fixture. Promotion requires a separate, explicitly authorized implementation request.


## Governed training bridge — training invocation readiness (Project Spec S0015)

This section calls the reusable `prepare_training_invocation_readiness`
governed training bridge from `pipeline/training.py`. It reports whether the
governed training entrypoint (`pipeline.training.train_from_paths`) is ready
to be invoked, using only two explicit path references — never this
notebook's in-memory `dataset_modeling_intent` or `execution_contract_draft`
objects, never notebook-held DataFrames, and never a hidden current working
directory. An `execution_contract_draft.v1` candidate (Project Spec S0014)
is always reported as not training-ready, with unresolved review items (for
example `TotalCharges` blank-value handling) surfaced explicitly rather than
silently accepted. This notebook never calls `train_from_paths` itself and
never stores hidden training state.

In [35]:
from pipeline.training import prepare_training_invocation_readiness

if write_local_output:
    # local_execution_contract_draft_output_path is only readable here
    # because write_local_output wrote it above; the bridge still receives
    # only the explicit path, not the in-memory execution_contract_draft
    # object. dataset_path here is the raw Telco CSV — this repository has
    # no separate "prepared dataset" output for Telco yet, so this call only
    # demonstrates the bridge's explicit-path readiness check; it is not a
    # claim that the raw CSV is a reviewed, prepared training dataset.
    training_invocation_readiness = prepare_training_invocation_readiness(
        local_execution_contract_draft_output_path,
        dataset_path,
    )
    print(json.dumps(training_invocation_readiness, indent=2))
    print()
    print("Blocking reasons before governed training could be invoked:")
    for reason in training_invocation_readiness["blocking_reasons"]:
        print(f"  - {reason}")
else:
    print(
        "write_local_output is False; no execution_contract_draft file exists "
        "on disk to reference explicitly, so the governed training bridge is "
        "not called here. Governed training is always invoked via "
        "pipeline.training.train_from_paths with an explicit, "
        "execution-ready execution_contract.v1 path and an explicit prepared "
        "dataset path — never from this notebook's in-memory "
        "dataset_modeling_intent or execution_contract_draft objects, "
        "notebook-held DataFrames, or a hidden current working directory."
    )

write_local_output is False; no execution_contract_draft file exists on disk to reference explicitly, so the governed training bridge is not called here. Governed training is always invoked via pipeline.training.train_from_paths with an explicit, execution-ready execution_contract.v1 path and an explicit prepared dataset path — never from this notebook's in-memory dataset_modeling_intent or execution_contract_draft objects, notebook-held DataFrames, or a hidden current working directory.


## Discovery evidence, preparation recipe, and prepared dataset materialization (Project Specs S0021/S0022/S0028/S0030)

Materialize the upstream handoff artifacts at repository-relative paths. The
raw dataset, discovery evidence, preparation recipe, prepared candidate CSV,
and prepared dataset metadata are all connected through reusable helpers in
`pipeline.prepare_candidate.py`; this notebook declares only the Telco policy
parameters and output references.

The approved `TotalCharges` blank-value policy resolved by Project Spec S0028
is applied by a reusable, tested boundary: null, empty-string, or
whitespace-only `TotalCharges` values are verified against `tenure == 0` for
every affected row, and only when that verification succeeds are the blanks
filled with `0.0` and the column normalized to numeric. When verification
passes, Project Spec S0030 allows this section to write
`pipeline/prepared/telco-customer-churn/prepared-data.csv` and refresh
`prepared-data-metadata.json` with a readable repository-relative reference.
If verification fails for a future dataset snapshot, no prepared candidate
reference is written and training readiness remains blocked. No model
training, release assembly, publisher operation, registry mutation, API
change, or UI change is performed by this section.

In [36]:
DISCOVERY_EVIDENCE_RELATIVE_PATH = f"pipeline/evidence/{dataset_slug}/discovery-evidence.json"
PREPARATION_RECIPE_RELATIVE_PATH = f"pipeline/evidence/{dataset_slug}/preparation-recipe.json"
PREPARED_DATA_METADATA_RELATIVE_PATH = f"pipeline/prepared/{dataset_slug}/prepared-data-metadata.json"
PREPARED_CANDIDATE_RELATIVE_PATH = f"pipeline/prepared/{dataset_slug}/prepared-data.csv"

# TotalCharges blank-value policy (Project Spec S0028). The verification,
# fill, prepared CSV write, and metadata reference/readiness checks live in
# reusable, tested helpers -- this notebook does not transform rows from
# hidden notebook-only DataFrame state.
TOTAL_CHARGES_PREPARATION_POLICY = {
    "column": "TotalCharges",
    "verification_column": "tenure",
    "verification_value": "0",
    "fill_value": "0.0",
    "description": (
        "Explicit approved TotalCharges preparation policy (Project Spec S0028): treat "
        "null, empty-string, or whitespace-only 'TotalCharges' values as missing; before "
        "filling, verify that every row with a blank 'TotalCharges' value has 'tenure' == 0; "
        "if and only if that verification succeeds for all blank rows, convert 'TotalCharges' "
        "to numeric and fill the verified blanks with 0.0; row dropping is not allowed by "
        "this policy; broad mean/median/mode imputation is not allowed by this policy; "
        "creating a missing-value indicator column is not allowed by this policy; if any "
        "blank 'TotalCharges' row fails verification (tenure != 0, non-numeric tenure, or "
        "otherwise), preparation is blocked and no prepared candidate is produced."
    ),
    "reason": (
        "Approved by Project Spec S0028 (Telco TotalCharges Preparation Policy Resolution), "
        "superseding the prior inferred_pending_review status. Verified against the real raw "
        "dataset (data/raw/telco-customer-churn.csv): all 11 observed blank 'TotalCharges' "
        "rows have 'tenure' == 0, confirming the fill is safe under this policy's "
        "verification predicate."
    ),
}

discovery_evidence = materialize_discovery_evidence(
    dataset_relative_path=dataset_relative_path,
    output_relative_path=DISCOVERY_EVIDENCE_RELATIVE_PATH,
    repo_root=_repo_root_path,
    dataset_slug=dataset_slug,
)

prepared_materialization = materialize_verified_conditional_fill_prepared_dataset(
    discovery_evidence_relative_path=DISCOVERY_EVIDENCE_RELATIVE_PATH,
    dataset_relative_path=dataset_relative_path,
    preparation_recipe_relative_path=PREPARATION_RECIPE_RELATIVE_PATH,
    prepared_candidate_relative_path=PREPARED_CANDIDATE_RELATIVE_PATH,
    prepared_metadata_relative_path=PREPARED_DATA_METADATA_RELATIVE_PATH,
    dataset_slug=dataset_slug,
    column=TOTAL_CHARGES_PREPARATION_POLICY["column"],
    verification_column=TOTAL_CHARGES_PREPARATION_POLICY["verification_column"],
    verification_value=TOTAL_CHARGES_PREPARATION_POLICY["verification_value"],
    fill_value=TOTAL_CHARGES_PREPARATION_POLICY["fill_value"],
    description=TOTAL_CHARGES_PREPARATION_POLICY["description"],
    reason=TOTAL_CHARGES_PREPARATION_POLICY["reason"],
    preparation_rules_source=(
        "pipeline.prepare_candidate:verified_conditional_fill_policy(TotalCharges)"
    ),
    repo_root=_repo_root_path,
)

preparation_recipe = prepared_materialization["preparation_recipe"]
prepared_data_metadata = prepared_materialization["prepared_data_metadata"]

assert discovery_evidence["schema_version"] == "dataset-discovery-evidence.v1"
assert discovery_evidence["dataset_metadata"]["source_path"] == dataset_relative_path
assert preparation_recipe["schema_version"] == "candidate-preparation-recipe.v1"
assert preparation_recipe["transformations"][0]["review_status"] == "inferred_approved"
assert prepared_data_metadata["schema_version"] == "prepared-data-metadata.v1"
assert prepared_data_metadata["dataset_identity"]["dataset_slug"] == dataset_slug
assert prepared_data_metadata["dataset_identity"]["raw_dataset_ref"]["path"] == dataset_relative_path
assert not any(preparation_recipe["preparation_boundary_confirmations"].values())
assert not any(prepared_data_metadata["materialization_boundary_confirmations"].values())

if preparation_recipe["candidate_output"]["produced"]:
    candidate_ref = prepared_data_metadata["prepared_candidate"]["reference"]
    assert candidate_ref is not None
    assert candidate_ref["path"] == PREPARED_CANDIDATE_RELATIVE_PATH
    assert candidate_ref["row_count"] == structure["row_count"]
    assert candidate_ref["column_count"] == structure["column_count"]
    assert prepared_data_metadata["ordered_prepared_columns"] == raw_columns
    assert prepared_data_metadata["training_readiness"]["is_training_ready"] is True
else:
    assert prepared_data_metadata["prepared_candidate"]["reference"] is None
    assert prepared_data_metadata["prepared_candidate"]["reason_not_produced"]
    assert prepared_data_metadata["training_readiness"]["is_training_ready"] is False

print(f"Discovery evidence written to: {DISCOVERY_EVIDENCE_RELATIVE_PATH}")
print(f"Preparation recipe written to: {PREPARATION_RECIPE_RELATIVE_PATH}")
print(f"Prepared dataset metadata written to: {PREPARED_DATA_METADATA_RELATIVE_PATH}")
print(f"Prepared candidate path: {prepared_materialization['prepared_candidate_path']}")
print(f"Prepared candidate produced: {prepared_data_metadata['prepared_candidate']['produced']}")
print(f"Training-ready: {prepared_data_metadata['training_readiness']['is_training_ready']}")
print(f"Training readiness reason: {prepared_data_metadata['training_readiness']['reason']}")
print(f"Unresolved review items: {len(prepared_data_metadata['unresolved_review_items'])}")

Discovery evidence written to: pipeline/evidence/telco-customer-churn/discovery-evidence.json
Preparation recipe written to: pipeline/evidence/telco-customer-churn/preparation-recipe.json
Prepared dataset metadata written to: pipeline/prepared/telco-customer-churn/prepared-data-metadata.json
Prepared candidate path: pipeline/prepared/telco-customer-churn/prepared-data.csv
Prepared candidate produced: True
Training-ready: True
Training readiness reason: None
Unresolved review items: 0


## Prepared dataset metadata summary

The metadata object above is the prepared dataset handoff record. It must only
report training readiness when `prepared_candidate.reference` points to a
readable repository-relative file under `pipeline/prepared/telco-customer-churn/`
and that file's row and column counts match the preparation recipe. The
prepared candidate is still not a final release candidate, publisher input,
registry artifact, API fixture, or UI fixture.

In [37]:
assert prepared_data_metadata["prepared_candidate"]["produced"] == preparation_recipe["candidate_output"]["produced"]
assert prepared_data_metadata["training_readiness"]["is_final_training_input"] is False

candidate_ref = prepared_data_metadata["prepared_candidate"]["reference"]
if prepared_data_metadata["prepared_candidate"]["produced"]:
    assert candidate_ref is not None
    candidate_path = _repo_root_path / candidate_ref["path"]
    assert candidate_path.exists()
    assert candidate_path.is_file()
    assert not Path(candidate_ref["path"]).is_absolute()
else:
    assert candidate_ref is None

print("Prepared metadata candidate reference:")
print(json.dumps(prepared_data_metadata["prepared_candidate"], indent=2))

Prepared metadata candidate reference:
{
  "produced": true,
  "reason_not_produced": null,
  "reference": {
    "path": "pipeline/prepared/telco-customer-churn/prepared-data.csv",
    "row_count": 7043,
    "column_count": 21,
    "content_sha256": "d4ac02e925c6a95b4d6d053ea61a4d0f3c845d85a8d8c85a511389eea3aded88"
  }
}


## Dataset modeling intent persistence — `dataset_modeling_intent.v1` (Project Spec S0023, updated by S0028)

Persist the `dataset_modeling_intent` object assembled earlier in this notebook
to a repository-relative path via the reusable `materialize_dataset_modeling_intent`
helper from `pipeline/discovery_evidence.py`. This step runs after the discovery
evidence, preparation recipe, and prepared dataset metadata materialization
sections above so the persisted artifact can reference each of those upstream
artifacts by explicit repository-relative path, plus the existing public
context artifact, when present. `TotalCharges` blank-value policy is
re-derived from the actual preparation recipe transformation on disk, so it
now reports `inferred_approved` (Project Spec S0028) instead of
`unresolved_pending_review` -- this notebook never decides that resolution on
its own authority, it only reflects whatever review status the preparation
recipe itself already recorded. The persisted artifact remains a
`dataset_modeling_intent.v1` authoring-intent object only: not an execution
contract, runtime contract, public contract, release candidate input,
publisher input, registry artifact, API fixture, or UI fixture. No model
training, contract promotion, release assembly, publisher operation, registry
mutation, API change, UI change, or notebook execution is authorized by this
section.


In [38]:
MODELING_INTENT_RELATIVE_PATH = f"pipeline/evidence/{dataset_slug}/dataset-modeling-intent.json"
PUBLIC_CONTEXT_RELATIVE_PATH = f"contracts/{dataset_slug}/dataset-context.json"

persisted_dataset_modeling_intent = materialize_dataset_modeling_intent(
    dataset_modeling_intent,
    output_relative_path=MODELING_INTENT_RELATIVE_PATH,
    repo_root=_repo_root_path,
    discovery_evidence_relative_path=DISCOVERY_EVIDENCE_RELATIVE_PATH,
    preparation_recipe_relative_path=PREPARATION_RECIPE_RELATIVE_PATH,
    prepared_data_metadata_relative_path=PREPARED_DATA_METADATA_RELATIVE_PATH,
    public_context_relative_path=PUBLIC_CONTEXT_RELATIVE_PATH,
)

assert persisted_dataset_modeling_intent["contract_version"] == "dataset_modeling_intent.v1"
assert persisted_dataset_modeling_intent["authoring_source"]["reduced_discovery_evidence_ref"] == (
    DISCOVERY_EVIDENCE_RELATIVE_PATH
)
assert persisted_dataset_modeling_intent["authoring_source"]["preparation_recipe_ref"] == (
    PREPARATION_RECIPE_RELATIVE_PATH
)
assert "customerID" not in persisted_dataset_modeling_intent["initial_feature_candidates"]
assert persisted_dataset_modeling_intent["blank_value_policy_candidates"].get("TotalCharges") == (
    preparation_recipe["transformations"][0]["review_status"]
), (
    "TotalCharges blank-value policy must reflect the preparation recipe's own "
    "review_status exactly -- it is never resolved on this notebook's own authority."
)
assert not any(
    persisted_dataset_modeling_intent["modeling_intent_boundary_confirmations"].values()
), (
    "Persisted dataset_modeling_intent must not be treated as an "
    "execution/runtime/public contract, release candidate input, publisher "
    "input, registry artifact, API fixture, or UI fixture."
)

print(f"Dataset modeling intent persisted to: {MODELING_INTENT_RELATIVE_PATH}")
print(
    "authoring_source references: "
    f"{json.dumps(persisted_dataset_modeling_intent['authoring_source'], indent=2)}"
)
print(f"TotalCharges blank_value_policy_candidates: {persisted_dataset_modeling_intent['blank_value_policy_candidates'].get('TotalCharges')}")
print(f"Unresolved review items: {len(persisted_dataset_modeling_intent['unresolved_review_items'])}")


Dataset modeling intent persisted to: pipeline/evidence/telco-customer-churn/dataset-modeling-intent.json
authoring_source references: {
  "authoring_notebook_ref": "notebooks/datasets/telco-customer-churn/01_dataset_authoring.ipynb",
  "reduced_discovery_evidence_ref": "pipeline/evidence/telco-customer-churn/discovery-evidence.json",
  "preparation_recipe_ref": "pipeline/evidence/telco-customer-churn/preparation-recipe.json",
  "prepared_data_metadata_ref": "pipeline/prepared/telco-customer-churn/prepared-data-metadata.json",
  "public_context_ref": "contracts/telco-customer-churn/dataset-context.json"
}
TotalCharges blank_value_policy_candidates: inferred_approved
Unresolved review items: 0


## Execution contract materialization — `execution_contract.v1` (Project Spec S0024, updated by S0028)

Materialize the official Telco execution contract at
`contracts/telco-customer-churn/execution-contract.json` via the reusable
`materialize_execution_contract` helper from `pipeline/contract_derivation.py`,
built from the `dataset_modeling_intent` persisted above, the materialized
discovery evidence, and the materialized preparation recipe. `customerID` is
excluded as an identifier column. `TotalCharges` blank-value handling is now
recorded as `inferred_approved` in the preparation recipe on disk (Project
Spec S0028), so `pipeline.contract_derivation._unresolved_review_columns`
correctly includes it in `feature_columns` rather than moving it into
`ignored_columns` — that exclusion rule only ever fires for a column whose
recipe review_status is *not* `explicit`/`inferred_approved`. `SeniorCitizen`
keeps its discovery-evidence-grounded `boolean` feature type — a binary
indicator, not a generic numeric default. Per-feature types for the feature
candidates are grounded in the materialized discovery evidence's own
`inferred_type` field, using the same type-compatibility mapping already
enforced by `pipeline/validate_contract_consistency.py`, so the materialized
contract passes that consistency check by construction. Training-policy
fields with no reviewed upstream value (`categorical_encoding_policy`,
`numeric_handling`, `allowed_transformations`, `split_policy`,
`primary_metric`, `secondary_metrics`, `modeling_constraints`) are populated
with conservative, disclosed repository-standard defaults — never silently
invented — and every such default is listed explicitly in the accompanying
`execution_contract_materialization_evidence.v1` artifact, alongside the
target/positive-label policy, identifier exclusion policy, unresolved feature
exclusions (now empty for `TotalCharges`), per-feature type grounding, and
boundary confirmations that the schema-locked `execution_contract.v1` shape
(`additionalProperties: false`) has no room for. This step validates the
materialized contract against `contracts/execution-contract.schema.json` and
cross-checks it against the materialized discovery evidence via
`pipeline/validate_contract_consistency.check`. No runtime/public contract
derivation, model training, model family selection, inference bundle
generation, release-candidate assembly, publisher operation, registry
mutation, API change, or UI change is performed by this section —
`contracts/telco-customer-churn/runtime-contract.json` and
`public-contract.json` are separate, already-materialized artifacts (Project
Spec S0025) that this section does not regenerate; if a future spec
explicitly promotes them into scope, they would need to be re-derived from
this updated execution contract to include `TotalCharges` as well.


In [39]:
from pipeline.contract_derivation import materialize_execution_contract
from pipeline.validate_contract_consistency import check as check_contract_consistency

EXECUTION_CONTRACT_RELATIVE_PATH = f"contracts/{dataset_slug}/execution-contract.json"
EXECUTION_CONTRACT_MATERIALIZATION_EVIDENCE_RELATIVE_PATH = (
    f"pipeline/evidence/{dataset_slug}/execution-contract-materialization-evidence.json"
)

execution_contract_materialization = materialize_execution_contract(
    persisted_dataset_modeling_intent,
    discovery_evidence,
    output_relative_path=EXECUTION_CONTRACT_RELATIVE_PATH,
    repo_root=_repo_root_path,
    preparation_recipe=preparation_recipe,
    evidence_output_relative_path=EXECUTION_CONTRACT_MATERIALIZATION_EVIDENCE_RELATIVE_PATH,
    discovery_evidence_relative_path=DISCOVERY_EVIDENCE_RELATIVE_PATH,
    preparation_recipe_relative_path=PREPARATION_RECIPE_RELATIVE_PATH,
    prepared_data_metadata_relative_path=PREPARED_DATA_METADATA_RELATIVE_PATH,
    modeling_intent_relative_path=MODELING_INTENT_RELATIVE_PATH,
    public_context_relative_path=PUBLIC_CONTEXT_RELATIVE_PATH,
    raw_dataset_relative_path=dataset_relative_path,
)

execution_contract = execution_contract_materialization["execution_contract"]
execution_contract_materialization_evidence = execution_contract_materialization[
    "execution_contract_materialization_evidence"
]

assert execution_contract["contract_version"] == "execution_contract.v1"
assert execution_contract["target_column"] == "Churn"
assert "customerID" not in execution_contract["feature_columns"]
assert "customerID" in execution_contract["ignored_columns"]

# Safety boundary: a column must never be silently approved into
# feature_columns. This assertion is tied to the preparation recipe's own
# review_status on disk (not a hardcoded expectation), so it stays correct
# whether TotalCharges' policy is resolved or not, and will correctly flip
# back to excluding TotalCharges if that review_status ever regresses away
# from an approved status.
total_charges_review_status = next(
    t["review_status"]
    for t in preparation_recipe["transformations"]
    if t["transformation_type"] == "missing_value_handling"
    and "TotalCharges" in t["target_columns"]
)
if total_charges_review_status in ("explicit", "inferred_approved"):
    assert "TotalCharges" in execution_contract["feature_columns"], (
        "TotalCharges blank-value handling is approved "
        f"({total_charges_review_status!r}) but was excluded from feature_columns."
    )
    assert "TotalCharges" not in execution_contract["ignored_columns"]
else:
    assert "TotalCharges" not in execution_contract["feature_columns"], (
        "TotalCharges blank-value handling is still "
        f"{total_charges_review_status!r}; it must never be silently approved "
        "into feature_columns."
    )
    assert "TotalCharges" in execution_contract["ignored_columns"]

assert execution_contract["feature_definitions"]["SeniorCitizen"]["type"] == "boolean"

check_contract_consistency(
    _repo_root_path / EXECUTION_CONTRACT_RELATIVE_PATH,
    _repo_root_path / DISCOVERY_EVIDENCE_RELATIVE_PATH,
    repo_root=_repo_root_path,
)

print(f"Execution contract written to: {EXECUTION_CONTRACT_RELATIVE_PATH}")
print(
    "Execution contract materialization evidence written to: "
    f"{EXECUTION_CONTRACT_MATERIALIZATION_EVIDENCE_RELATIVE_PATH}"
)
print(f"feature_columns ({len(execution_contract['feature_columns'])}): {execution_contract['feature_columns']}")
print(f"ignored_columns: {execution_contract['ignored_columns']}")
print(f"TotalCharges review_status: {total_charges_review_status}")
print("Consistency check passed: execution contract is consistent with discovery evidence.")


Execution contract written to: contracts/telco-customer-churn/execution-contract.json
Execution contract materialization evidence written to: pipeline/evidence/telco-customer-churn/execution-contract-materialization-evidence.json
feature_columns (19): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
ignored_columns: ['customerID']
TotalCharges review_status: inferred_approved
Consistency check passed: execution contract is consistent with discovery evidence.


## Governed training run materialization — `training_run_materialization_result.v1` (Project Spec S0026)

This section calls the reusable `materialize_training_run_from_prepared_metadata`
governed bridge from `pipeline/training.py`. It requires an execution-ready
`execution_contract.v1` (never an `execution_contract_draft.v1`) and a
`prepared-data-metadata.v1` artifact whose `prepared_candidate.produced` and
`training_readiness.is_training_ready` are both true, with an explicit,
readable `prepared_candidate.reference` path -- never the raw Telco CSV, this
notebook's in-memory state, or a hidden current working directory.

Project Spec S0030 can make that prepared metadata training-ready, but this
notebook section remains only a governed bridge demonstration. No model
training, model family selection, release-candidate assembly, publisher
operation, registry mutation, API change, or UI change is performed unless a
separate implementation request explicitly authorizes the downstream training
run materialization scope.

In [40]:
from pipeline.training import materialize_training_run_from_prepared_metadata

telco_training_run_materialization = materialize_training_run_from_prepared_metadata(
    _repo_root_path / EXECUTION_CONTRACT_RELATIVE_PATH,
    _repo_root_path / PREPARED_DATA_METADATA_RELATIVE_PATH,
    dataset_slug=dataset_slug,
)

print(json.dumps(telco_training_run_materialization, indent=2))
print()
if telco_training_run_materialization["status"] == "blocked":
    print("Governed training run materialization is blocked:")
    for reason in telco_training_run_materialization["blocking_reasons"]:
        print(f"  - {reason}")
else:
    print(
        "Governed training run materialized at: "
        f"{telco_training_run_materialization['training_result']['output_directory']}"
    )

{
  "artifact_type": "training_run_materialization_result",
  "contract_version": "training_run_materialization_result.v1",
  "status": "trained",
  "execution_contract_identity": "execution_ready",
  "prepared_dataset_reference": "pipeline/prepared/telco-customer-churn/prepared-data.csv",
  "blocking_reasons": [],
  "governed_entrypoint": "pipeline.training.train_from_paths",
  "materialization_boundary_confirmations": {
    "model_training_performed": true,
    "training_run_materialized": true
  },
  "training_result": {
    "status": "trained",
    "model_family": "gradient_boosting",
    "task_type": "classification",
    "dataset_id": "telco-customer-churn",
    "target_column": "Churn",
    "feature_columns": [
      "gender",
      "SeniorCitizen",
      "Partner",
      "Dependents",
      "tenure",
      "PhoneService",
      "MultipleLines",
      "InternetService",
      "OnlineSecurity",
      "OnlineBackup",
      "DeviceProtection",
      "TechSupport",
      "StreamingT

## Telco inference-bundle materialization — `inference_bundle.v1` (Project Spec S0033)

This section calls the reusable `materialize_governed_inference_bundle`
boundary from `pipeline/generate_inference_bundle.py`. It only proceeds when
the governed training run materialization result computed above has
`status: "trained"` — never a hardcoded `train-pending` placeholder, a glob
over `pipeline/training-runs/`, or this notebook's own in-memory state. The
prepared dataset reference is resolved through the same
`prepared-data-metadata.json` boundary `pipeline.training` itself uses, so it
always matches the exact prepared dataset the training run was produced
from.

The materialized descriptor is written to
`contracts/telco-customer-churn/inference-bundle.json` and validated against
`contracts/inference-bundle.schema.json` before being persisted. Its
`release_context.release_id` is a deterministic placeholder tied to the
governed training run's own date (`_derive_provisional_release_id`) — this
spec does not assemble a release candidate, so no real release_id has been
allocated yet; a later, separately authorized release-candidate assembly
(Project Spec S0032) is free to supersede it. This section never assembles a
release candidate, creates a `publisher/runs/*` record, runs publisher
validation, promotes a release, activates registry state, or changes
API/UI/Docker/Dashboard behavior.

In [41]:
from pipeline.generate_inference_bundle import materialize_governed_inference_bundle

RUNTIME_CONTRACT_RELATIVE_PATH = f"contracts/{dataset_slug}/runtime-contract.json"
PUBLIC_CONTRACT_RELATIVE_PATH = f"contracts/{dataset_slug}/public-contract.json"
INFERENCE_BUNDLE_RELATIVE_PATH = f"contracts/{dataset_slug}/inference-bundle.json"

if telco_training_run_materialization["status"] != "trained":
    print(
        "Inference-bundle materialization is blocked: no governed training "
        "run is available yet."
    )
    for reason in telco_training_run_materialization["blocking_reasons"]:
        print(f"  - {reason}")
    telco_inference_bundle_materialization = {
        "status": "blocked",
        "blocking_reasons": [
            "telco_training_run_materialization.status is not 'trained'.",
        ],
    }
else:
    telco_inference_bundle_materialization = materialize_governed_inference_bundle(
        training_run_materialization_result=telco_training_run_materialization,
        execution_contract_path=_repo_root_path / EXECUTION_CONTRACT_RELATIVE_PATH,
        runtime_contract_path=_repo_root_path / RUNTIME_CONTRACT_RELATIVE_PATH,
        public_contract_path=_repo_root_path / PUBLIC_CONTRACT_RELATIVE_PATH,
        dataset_context_path=_repo_root_path / PUBLIC_CONTEXT_RELATIVE_PATH,
        prepared_data_metadata_path=_repo_root_path / PREPARED_DATA_METADATA_RELATIVE_PATH,
        output_path=_repo_root_path / INFERENCE_BUNDLE_RELATIVE_PATH,
        repo_root=_repo_root_path,
        prediction_type="string",
        class_labels=["No", "Yes"],
        probability_output=True,
        execution_contract_ref=EXECUTION_CONTRACT_RELATIVE_PATH,
        runtime_contract_ref=RUNTIME_CONTRACT_RELATIVE_PATH,
        public_contract_ref=PUBLIC_CONTRACT_RELATIVE_PATH,
        dataset_context_ref=PUBLIC_CONTEXT_RELATIVE_PATH,
    )
    print(json.dumps(telco_inference_bundle_materialization, indent=2))
    print()
    if telco_inference_bundle_materialization["status"] == "blocked":
        print("Inference-bundle materialization is blocked:")
        for reason in telco_inference_bundle_materialization["blocking_reasons"]:
            print(f"  - {reason}")
    else:
        print(
            "Inference-bundle descriptor materialized at: "
            f"{telco_inference_bundle_materialization['output_path']}"
        )

{
  "status": "generated",
  "output_path": "/home/fabyuu/Projetos/N8N/atlas-dataflow/contracts/telco-customer-churn/inference-bundle.json",
  "bundle_id": "telco-customer-churn-inference-bundle-20260710T191735Z",
  "training_run_id": "train-20260710T191735Z",
  "provisional_release_id": "release-20260710-001",
  "prepared_dataset_reference": "pipeline/prepared/telco-customer-churn/prepared-data.csv"
}

Inference-bundle descriptor materialized at: /home/fabyuu/Projetos/N8N/atlas-dataflow/contracts/telco-customer-churn/inference-bundle.json


## Release-candidate data handoff readiness — `release-candidate-handoff-readiness.v1` (Project Spec S0016, updated by S0031)

This section calls the reusable `build_release_candidate_handoff_readiness`
pre-assembly handoff boundary from `pipeline/assemble_candidate.py`. It
reports whether an explicit, repository-relative artifact reference set —
one path per `release-candidate-input.v1` required role (discovery
evidence, execution contract, runtime contract, public contract,
preparation recipe, prepared data metadata, training parameter record,
model artifact, training metrics, model card, public context, inference
bundle) — is ready to be handed off toward a future, separately governed
release-candidate assembly (`pipeline/assemble_candidate.py`'s own
`main()`), publisher validation, publisher promotion, registry activation,
API serving, or UI data-fill. Each of those six later stages is explicitly
confirmed as not performed by this readiness check
(`handoff_boundary_confirmations`).

The four training-related roles below (`training_parameter_record`,
`model_artifact`, `training_metrics`, `model_card`) are normalized from the
governed training run materialization result computed above via the
reusable `normalize_training_handoff_references` boundary in
`pipeline/assemble_candidate.py` (Project Spec S0031). When that result has
`status: "trained"`, the explicit repository-relative paths reported by
`training_result` are used; otherwise the `train-pending` placeholder paths
below are kept as the pre-training blocked-state fallback and are never
reported ready, since the placeholder directory never exists. The other
paths below are the conventional locations these M22-M25 governed
artifacts are expected to occupy once Telco has gone through real
discovery-evidence and contract-promotion stages. The existing
`contracts/telco-customer-churn/dataset-context.json` file should be reported
ready as `public_context` when the repository root is resolved correctly;
other upstream artifacts that have not been produced yet should remain
`missing_reference`. This notebook never derives an artifact reference from
its own in-memory `dataset_modeling_intent`, `execution_contract_draft`, or
`training_invocation_readiness` objects, never infers a training run by
globbing `pipeline/training-runs/`, and never calls release-candidate
assembly, publisher validation, publisher promotion, or registry activation
itself.

In [42]:
from pipeline.assemble_candidate import (
    build_release_candidate_handoff_readiness,
    normalize_training_handoff_references,
)

# Conventional, repository-relative locations for Telco's M22-M25 governed
# artifacts. Existing repository artifacts should report ready; upstream
# artifacts that have not been produced yet should remain missing_reference.
# The train-pending paths below are the pre-training blocked-state fallback
# for the four training-related roles -- they are only used when the
# governed training run materialization result above does not have
# status == "trained"; that placeholder directory never exists, so it is
# never reported ready.
TELCO_TRAIN_PENDING_TRAINING_ROLE_PATHS = {
    "training_parameter_record": (
        f"pipeline/training-runs/{dataset_slug}/train-pending/training-parameter-record.json"
    ),
    "model_artifact": f"pipeline/training-runs/{dataset_slug}/train-pending/model.pkl",
    "training_metrics": f"pipeline/training-runs/{dataset_slug}/train-pending/metrics.json",
    "model_card": f"pipeline/training-runs/{dataset_slug}/train-pending/model-card.json",
}

telco_training_handoff_normalization = normalize_training_handoff_references(
    telco_training_run_materialization,
    repo_root=_repo_root_path,
)
telco_training_handoff_role_paths = {
    role: telco_training_handoff_normalization["role_paths"].get(role, placeholder_path)
    for role, placeholder_path in TELCO_TRAIN_PENDING_TRAINING_ROLE_PATHS.items()
}

TELCO_RELEASE_CANDIDATE_ARTIFACT_REFERENCES = {
    "discovery_evidence": f"pipeline/evidence/{dataset_slug}/discovery-evidence.json",
    "execution_contract": f"contracts/{dataset_slug}/execution-contract.json",
    "runtime_contract": f"contracts/{dataset_slug}/runtime-contract.json",
    "public_contract": f"contracts/{dataset_slug}/public-contract.json",
    "preparation_recipe": f"pipeline/evidence/{dataset_slug}/preparation-recipe.json",
    "prepared_data_metadata": f"pipeline/prepared/{dataset_slug}/prepared-data-metadata.json",
    **telco_training_handoff_role_paths,
    "public_context": f"contracts/{dataset_slug}/dataset-context.json",
    "inference_bundle": f"contracts/{dataset_slug}/inference-bundle.json",
}

release_candidate_handoff_readiness = build_release_candidate_handoff_readiness(
    TELCO_RELEASE_CANDIDATE_ARTIFACT_REFERENCES,
    repo_root=_repo_root_path,
)

print(json.dumps(release_candidate_handoff_readiness, indent=2))
print()
path_normalization_roles = {
    result["role"]
    for result in release_candidate_handoff_readiness["role_results"]
    if result["reason"] in {"absolute_path_rejected", "parent_traversal_rejected"}
}
missing_reference_roles = {
    result["role"]
    for result in release_candidate_handoff_readiness["role_results"]
    if result["reason"] == "missing_reference"
}
print("Release-candidate handoff readiness summary:")
if missing_reference_roles:
    print("  Expected missing upstream artifact references:")
    for role in sorted(missing_reference_roles):
        print(f"    - {role}")
if path_normalization_roles:
    print("  Path/root normalization problems requiring notebook correction:")
    for role in sorted(path_normalization_roles):
        print(f"    - {role}")
if not release_candidate_handoff_readiness["blocking_reasons"]:
    print("  All explicit release-candidate handoff references are ready.")

{
  "schema_version": "release-candidate-handoff-readiness.v1",
  "handoff_kind": "release_candidate_data_handoff",
  "required_roles": [
    "discovery_evidence",
    "execution_contract",
    "runtime_contract",
    "public_contract",
    "preparation_recipe",
    "prepared_data_metadata",
    "training_parameter_record",
    "model_artifact",
    "training_metrics",
    "model_card",
    "public_context",
    "inference_bundle"
  ],
  "role_results": [
    {
      "role": "discovery_evidence",
      "path": "pipeline/evidence/telco-customer-churn/discovery-evidence.json",
      "ready": true,
      "reason": null
    },
    {
      "role": "execution_contract",
      "path": "contracts/telco-customer-churn/execution-contract.json",
      "ready": true,
      "reason": null
    },
    {
      "role": "runtime_contract",
      "path": "contracts/telco-customer-churn/runtime-contract.json",
      "ready": true,
      "reason": null
    },
    {
      "role": "public_contract",
      "p

## Release-candidate assembly from governed training run — `release-candidate-input.v1` (Project Spec S0032)

This section only runs when the release-candidate handoff readiness check
above reports `is_release_candidate_input_ready: True`. It never assembles
a release candidate from a blocked handoff, a notebook-held DataFrame, or
any path not already classified `ready` by
`build_release_candidate_handoff_readiness`.

When ready, it derives a deterministic `release_id` from the governed
training run id (`derive_deterministic_release_id`, Project Spec S0032) --
never a milestone tag, notebook counter, or reused static fixture id --
then builds a schema-compliant `release-candidate-input.v1` payload
(`build_release_candidate_input`) from the same explicit artifact
references used for the handoff readiness check above, and assembles the
resulting release candidate package (`assemble_release_candidate`, reused
from `pipeline/assemble_candidate.py`'s own CLI logic) under
`releases/candidates/telco-customer-churn/{release_id}/`.

This does not create a `publisher/runs/*` record, does not run publisher
promotion, does not activate registry state, and does not change API, UI,
or Docker behavior -- it only assembles a candidate package and validates
it against the existing publisher candidate boundary
(`publisher/validate.py`).

In [43]:
from pipeline.assemble_candidate import (
    assemble_release_candidate,
    build_release_candidate_input,
    derive_deterministic_release_id,
)

if not release_candidate_handoff_readiness["is_release_candidate_input_ready"]:
    print("Release-candidate assembly is blocked: handoff readiness is not satisfied.")
    for reason in release_candidate_handoff_readiness["blocking_reasons"]:
        print(f"  - {reason}")
    telco_release_candidate_input = None
    telco_release_candidate_assembly_result = None
else:
    telco_training_run_id = Path(
        telco_training_run_materialization["training_result"]["output_directory"]
    ).name

    telco_release_id = derive_deterministic_release_id(telco_training_run_id)

    telco_release_candidate_input = build_release_candidate_input(
        dataset_slug=dataset_slug,
        release_id=telco_release_id,
        source_run_id=telco_training_run_id,
        artifact_references=TELCO_RELEASE_CANDIDATE_ARTIFACT_REFERENCES,
        repo_root=_repo_root_path,
        producer=str(NOTEBOOK_RELATIVE_PATH),
    )

    telco_release_candidate_assembly_result = assemble_release_candidate(
        telco_release_candidate_input,
        _repo_root_path / "releases" / "candidates",
        repo_root=_repo_root_path,
        source_input_label=f"{dataset_slug}-{telco_release_id}-release-candidate-input",
    )

    print(json.dumps(telco_release_candidate_assembly_result, indent=2))
    print()
    if telco_release_candidate_assembly_result["status"] == "accepted":
        print(
            "Release candidate assembled at: "
            f"{telco_release_candidate_assembly_result['candidate_dir']}"
        )
    else:
        print("Release-candidate assembly was rejected:")
        print(f"  phase : {telco_release_candidate_assembly_result['rejection_phase']}")
        print(f"  reason: {telco_release_candidate_assembly_result['reason']}")

{
  "status": "accepted",
  "dataset_slug": "telco-customer-churn",
  "release_id": "release-20260710t191735z",
  "candidate_dir": "/home/fabyuu/Projetos/N8N/atlas-dataflow/releases/candidates/telco-customer-churn/release-20260710t191735z",
  "publisher_validation": {
    "valid": true,
    "role_results": {
      "contracts": {
        "role": "contracts",
        "status": "present",
        "required": true,
        "artifact_reference": "contracts/runtime-contract.json",
        "declared_sha256_present": false
      },
      "predictive_bundle": {
        "role": "predictive_bundle",
        "status": "present",
        "required": true,
        "artifact_reference": "predictions/bundle.json",
        "declared_sha256_present": false
      },
      "metrics": {
        "role": "metrics",
        "status": "present",
        "required": true,
        "artifact_reference": "metrics/metrics.json",
        "declared_sha256_present": false
      },
      "model_card": {
        "role":

## Publisher-validation run materialization from release candidate — `release-candidate-validation.v1` (Project Spec S0034)

This section calls `materialize_telco_validation_run`, a narrow,
Telco-only boundary added to `publisher/validate.py`, that identifies the
just-assembled release candidate from `telco_release_candidate_assembly_result`
above, validates it via the existing `publisher/validate.py` boundary
(`run()`), and generates `manifest.json` via the existing
`publisher/manifest.py` boundary (`run()`) in the same run directory only
when the validation result's own `promotion_gate.promotion_allowed` is
`true`.

This step runs only when release-candidate assembly above reported
`status == "accepted"`; it is skipped with a clear message when assembly
was blocked, rejected, or never executed. It never infers a candidate from
notebook memory or a glob over `releases/candidates/`, and it rejects any
candidate whose resolved `dataset_slug` is not `telco-customer-churn`.

It never promotes a release, never activates or updates
`registry/datasets.json`, and never modifies any release-candidate
artifact -- those boundaries are enforced inside
`materialize_telco_validation_run` itself (`boundary_confirmations`), not
just documented here. The resulting `publisher/runs/validate-*/` directory
is discoverable by the existing `/api/admin/runs` reader through the same
`publisher/runs/` convention already used by every other publisher run, so
no API or UI change is required for the Admin/Dashboard run list to show
it.

In [44]:
from publisher.validate import materialize_telco_validation_run

if (
    telco_release_candidate_assembly_result is None
    or telco_release_candidate_assembly_result.get("status") != "accepted"
):
    print(
        "Publisher-validation run materialization is skipped: no accepted "
        "Telco release candidate is available from the assembly step above."
    )
    telco_publisher_validation_materialization = None
else:
    telco_publisher_validation_materialization = materialize_telco_validation_run(
        telco_release_candidate_assembly_result,
        repo_root=_repo_root_path,
    )

    print(json.dumps(telco_publisher_validation_materialization, indent=2))
    print()
    if telco_publisher_validation_materialization["materialization_status"] == "materialized":
        print(
            "Publisher validation run materialized at: "
            f"publisher/runs/{telco_publisher_validation_materialization['run_id']}"
        )
        print(f"  validation_outcome : {telco_publisher_validation_materialization['validation_outcome']}")
        print(f"  manifest_generated : {telco_publisher_validation_materialization['manifest_generated']}")
    else:
        print("Publisher-validation run materialization was blocked:")
        print(f"  reason_code: {telco_publisher_validation_materialization['reason_code']}")
        print(f"  message    : {telco_publisher_validation_materialization['message']}")

{
  "materialization_status": "materialized",
  "reason_code": null,
  "message": null,
  "run_id": "validate-20260710T191735Z",
  "run_dir": "publisher/runs/validate-20260710T191735Z",
  "dataset_slug": "telco-customer-churn",
  "release_id": "release-20260710t191735z",
  "validation_outcome": "accepted",
  "manifest_generated": true,
  "manifest_path": "publisher/runs/validate-20260710T191735Z/manifest.json",
  "manifest_error": null,
  "boundary_confirmations": {
    "publisher_promotion_performed": false,
    "registry_activation_performed": false,
    "release_candidate_artifact_modified": false
  }
}

Publisher validation run materialized at: publisher/runs/validate-20260710T191735Z
  validation_outcome : accepted
  manifest_generated : True
